# Titanic Data Preprocessing

This notebook follows the recommended sequence: duplicates, missing values, outliers, feature encoding, and scaling. It uses the Titanic training CSV included in this repository.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [ ]:
# Support running from either this notebook's folder or the repository root.
data_candidates = [
    Path("files") / "titanic dataset" / "train.csv",
    Path("Week_2") / "Day_2" / "files" / "titanic dataset" / "train.csv",
]
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find files/titanic dataset/train.csv")

train_df = pd.read_csv(data_path)
print(f"Loaded {len(train_df)} rows and {len(train_df.columns)} columns from {data_path}.")
display(train_df.head())

## Exercise 1: Find and remove duplicate rows

Only exact duplicate rows are removed. Compare the row counts before and after, then verify no duplicates remain.

In [ ]:
rows_before = len(train_df)
duplicate_count = int(train_df.duplicated().sum())
train_df = train_df.drop_duplicates().copy()
rows_after = len(train_df)

print(f"Duplicate rows found: {duplicate_count}")
print(f"Rows before: {rows_before}; rows after: {rows_after}; rows removed: {rows_before - rows_after}")
assert rows_after == rows_before - duplicate_count

In [ ]:
print("Remaining duplicate rows:", int(train_df.duplicated().sum()))

## Exercise 2: Compare and handle missing values

The table compares dropping rows, statistical imputation, and constant filling for Age, Embarked, and Cabin. The selected choices are median Age (robust to skew and outliers), dropping the few rows missing Embarked, and filling Cabin with "Unknown" because Cabin is missing for many passengers and the missingness itself is informative.

In [ ]:
missing_columns = ["Age", "Embarked", "Cabin"]
missing_counts = train_df[missing_columns].isna().sum()
row_count = len(train_df)

strategy_comparison = pd.DataFrame([
    {
        "Column": "Age",
        "Missing values": int(missing_counts["Age"]),
        "Rows kept if dropped": row_count - int(missing_counts["Age"]),
        "Statistical imputation": "Median; robust to skew and outliers",
        "Constant fill": "0 is misleading because it creates newborn ages",
    },
    {
        "Column": "Embarked",
        "Missing values": int(missing_counts["Embarked"]),
        "Rows kept if dropped": row_count - int(missing_counts["Embarked"]),
        "Statistical imputation": "Mode; may treat an estimate as observed data",
        "Constant fill": "Unknown preserves missingness but adds a category",
    },
    {
        "Column": "Cabin",
        "Missing values": int(missing_counts["Cabin"]),
        "Rows kept if dropped": row_count - int(missing_counts["Cabin"]),
        "Statistical imputation": "Mode is misleading for high-cardinality cabin IDs",
        "Constant fill": "Unknown preserves missingness and retains rows",
    },
]).set_index("Column")

display(strategy_comparison)

In [ ]:
train_df["Age"] = train_df["Age"].fillna(train_df["Age"].median())

# Only a small number of rows lack Embarked, so dropping them avoids inventing a port.
rows_before_embarked_drop = len(train_df)
train_df = train_df.dropna(subset=["Embarked"]).copy()
print(f"Rows dropped for missing Embarked: {rows_before_embarked_drop - len(train_df)}")

# Cabin is missing for many passengers; Unknown keeps those rows and marks missingness.
train_df["Cabin"] = train_df["Cabin"].fillna("Unknown")
display(train_df[missing_columns].isna().sum().rename("Remaining missing values").to_frame())

## Exercise 4: Visualize and treat outliers

Inspect Fare and Age distributions before treatment. The 1.5 × IQR fences are a standard exploratory rule, not a claim that high fares or ages are invalid. Cap values at the fences rather than deleting passengers, which preserves the training rows.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for column, row in [("Fare", 0), ("Age", 1)]:
    values = train_df[column].dropna()
    axes[row, 0].hist(values, bins=30, edgecolor="black")
    axes[row, 0].set_title(f"{column} distribution")
    axes[row, 0].set_xlabel(column)
    axes[row, 1].boxplot(values, vert=False)
    axes[row, 1].set_title(f"{column} boxplot")
    axes[row, 1].set_xlabel(column)

plt.tight_layout()
plt.show()

In [ ]:
iqr_bounds = {}
outlier_summary = []

for column in ["Fare", "Age"]:
    q1 = train_df[column].quantile(0.25)
    q3 = train_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (train_df[column] < lower_bound) | (train_df[column] > upper_bound)
    outlier_count = int(outlier_mask.sum())

    iqr_bounds[column] = (lower_bound, upper_bound)
    outlier_summary.append({
        "Column": column,
        "Lower IQR fence": lower_bound,
        "Upper IQR fence": upper_bound,
        "Outliers detected": outlier_count,
    })

    # Capping limits extreme values while retaining each passenger in the data.
    train_df[column] = train_df[column].clip(lower=lower_bound, upper=upper_bound)

display(pd.DataFrame(outlier_summary).set_index("Column"))

In [ ]:
for column, (lower_bound, upper_bound) in iqr_bounds.items():
    remaining = int(((train_df[column] < lower_bound) | (train_df[column] > upper_bound)).sum())
    print(f"{column}: {remaining} values remain outside the IQR fences after capping")

## Exercises 3 and 6: Extract titles and encode categorical features

Title is extracted from Name. One-hot encoding is used for nominal features because their categories have no natural order; the encoded DataFrames are concatenated back into `train_df`. Pclass is ordinal (1st, 2nd, 3rd), so label encoding preserves its rank as 0, 1, 2.

In [ ]:
train_df["Title"] = train_df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
train_df["FamilySize"] = train_df["SibSp"] + train_df["Parch"] + 1

# Keep a compact signal from Cabin before dropping the high-cardinality cabin IDs.
train_df["HasCabin"] = train_df["Cabin"].ne("Unknown").astype(int)
display(train_df[["Name", "Title", "FamilySize", "Cabin", "HasCabin"]].head())

In [ ]:
# Encode Title, then merge the generated columns into the main DataFrame.
title_encoder = OneHotEncoder(handle_unknown="ignore")
encoded_titles = title_encoder.fit_transform(train_df[["Title"]])
if hasattr(encoded_titles, "toarray"):
    encoded_titles = encoded_titles.toarray()
encoded_title_columns = title_encoder.get_feature_names_out(["Title"])
encoded_titles_df = pd.DataFrame(
    encoded_titles,
    columns=encoded_title_columns,
    index=train_df.index,
)
train_df = pd.concat([train_df.drop(columns=["Title"]), encoded_titles_df], axis=1)

# Sex and Embarked are nominal too, so one-hot encode and merge them as well.
nominal_columns = ["Sex", "Embarked"]
nominal_encoder = OneHotEncoder(handle_unknown="ignore")
encoded_nominal = nominal_encoder.fit_transform(train_df[nominal_columns])
if hasattr(encoded_nominal, "toarray"):
    encoded_nominal = encoded_nominal.toarray()
nominal_feature_columns = nominal_encoder.get_feature_names_out(nominal_columns)
encoded_nominal_df = pd.DataFrame(
    encoded_nominal,
    columns=nominal_feature_columns,
    index=train_df.index,
)
train_df = pd.concat([train_df.drop(columns=nominal_columns), encoded_nominal_df], axis=1)

print(f"Added {len(encoded_title_columns)} Title, {len(nominal_feature_columns)} Sex/Embarked one-hot columns.")

In [ ]:
# Pclass is an ordered feature; LabelEncoder maps 1 < 2 < 3 to 0 < 1 < 2.
pclass_encoder = LabelEncoder()
train_df["Pclass"] = pclass_encoder.fit_transform(train_df["Pclass"])
pclass_mapping = {
    original: int(encoded)
    for original, encoded in zip(pclass_encoder.classes_, pclass_encoder.transform(pclass_encoder.classes_))
}
print("Pclass order-preserving mapping:", pclass_mapping)

## Exercise 7: Bin Age and one-hot encode the groups

Use interpretable boundaries: child (0–12], teen (12–19], adult (19–60], and senior (>60). These are descriptive bins for this exercise, not clinical or universal age definitions.

In [ ]:
age_bins = [0, 12, 19, 60, float("inf")]
age_labels = ["child", "teen", "adult", "senior"]
train_df["AgeGroup"] = pd.cut(
    train_df["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
)

age_group_features = pd.get_dummies(train_df["AgeGroup"], prefix="AgeGroup", dtype=int)
train_df = pd.concat([train_df.drop(columns=["AgeGroup"]), age_group_features], axis=1)
print("Age-group counts:")
display(pd.cut(train_df["Age"], bins=age_bins, labels=age_labels, include_lowest=True).value_counts().sort_index())

## Exercise 5: Standardize numerical features

Scale the numeric predictors after outlier capping. StandardScaler centers them at zero and scales to unit variance. Do not scale the binary target, passenger identifier, or one-hot indicator columns.

In [ ]:
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize"]
scaler = StandardScaler()
train_df[numeric_features] = scaler.fit_transform(train_df[numeric_features])

display(train_df[numeric_features].agg(["mean", "std"]).round(3))

In [ ]:
# Remove raw text identifiers; retain HasCabin instead of high-cardinality Cabin codes.
model_df = train_df.drop(columns=["Name", "Ticket", "Cabin"])
X = model_df.drop(columns=["Survived", "PassengerId"])
y = model_df["Survived"]

print("Preprocessed training data:", train_df.shape)
print("Model feature matrix:", X.shape)
print("Target shape:", y.shape)
print("Remaining missing values:", int(train_df.isna().sum().sum()))
print("Non-numeric model features:", list(X.select_dtypes(exclude="number").columns))
display(X.head())